# Assignment 1

![](https://media.giphy.com/media/xT9C25UNTwfZuk85WP/giphy-downsized-large.gif)

Remember the rules of ~Fight~ Code Club:
1. ALWAYS DOCUMENT
2. Cite resources that you use (paste links)
3. Include the names people who you worked with
4. Be neat and organized

## Scrape and Clean data

Based on you proposal, scrape or collect your data:

1. One variable must be either: (40 pts)
    1. scraped from the web OR;
    2. collected from an API AND you must create one new variable that is "new" to the best of your knowledge (combination of other variables representing something new).

2. You must have at least 3 variables, but you may include as many as you want into you final dataset. Likely, you will want to include more to make graphs and regressions. (30pts)

3. You must *be able* to run a regression that makes some sense with this data (the regression doesn't have to be a complete model). Briefly describe one regression you would run with your variables. (**DO NOT** run a regression, yet). (15pts)

4. You must have one combined and cleaned dataset (15 pts)

You must submit one python notebook on how you scraped/gathered data from an api, and how you combined and cleaned you data. I should be able to run your code and reproduce your final data set.  

The other variables that you choose to include do not have to be collected by API or webscraped, but you do have to combine the files and clean the dataset with python.

Thus, you must submit:
- Your finalized data set (only one) (note: you may add more variables in the future).
- Your documented python notebook
- Any associated data files needed to produce the final dataset.

You will be evaluated on:
- Completeness of the data
- Quality of the code 
- The creativity of the new variable/webscraped data you gather

Be sure to upload ALL associated files for your code to run. I will run your code from beginning to end - make it easy for me to replicate your code.

In [142]:
import requests
import pandas as pd
import numpy as np
import pprint
from bs4 import BeautifulSoup

import os 

from datetime import date
import time
from time import sleep
import selenium #Need this step 
from selenium import webdriver #Need this step 

from selenium.webdriver.common.by import By #Allows for selenium to click things 
from selenium.webdriver.chrome.service import Service #https://stackoverflow.com/questions/64717302/deprecationwarning-executable-path-has-been-deprecated-selenium-python
from selenium.webdriver.support import expected_conditions as EC #Allows for more complex code 
from selenium.webdriver.chrome.options import Options #Allows you to change aspects of the browser

In [143]:
from boxoffice_api import BoxOffice 

In [169]:
chrome_options = Options() 

driver = webdriver.Chrome()
chrome_options.add_argument("--window-size=1900,1000") #pick a size and stick with it, it can change how code intreacts with the website.
driver = webdriver.Chrome(options = chrome_options)

IMDb link to the list I created by myself: https://www.imdb.com/list/ls4173593959/?ref_=lsedt_bk

The list includes films released before 9/11, 2001. 


In [145]:


url = "https://www.imdb.com/list/ls4173593959/?ref_=lsedt_bk" 
 
driver.get(url)
driver.implicitly_wait(10)

buckets = driver.find_elements(By.XPATH, "//li[contains(@class, 'ipc-metadata-list-summary-item')]")

titles = []
years = []
guides = []
ratings = []
movie_ids = []

count = 1

for bucket in buckets:
    title_element = bucket.find_element(By.XPATH, ".//h3") # getting the titles
    clean_title = title_element.text.split(". ", 1)[1]  # cleaning tags

    year_element = bucket.find_element(By.XPATH, f'/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/div/ul/span[1]')
    clean_year = year_element.text
    
    guide_element = bucket.find_element(By.XPATH, f'/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/div/ul/span[3]')
    clean_guide = guide_element.text

    rating_element = bucket.find_element(By.XPATH, f'/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/span/div/span/span[1]')
    clean_rating = rating_element.text
    
    titles.append(clean_title)
    years.append(clean_year)
    guides.append(clean_guide)
    ratings.append(clean_rating)

    # getting movie ids to find the reviews
    id_element = bucket.find_element(By.XPATH, f"/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/ul/div/a").get_attribute('href')
    clean_id = id_element.split('/')[4]
    movie_ids.append(clean_id)

    count += 1
    
ids_df = pd.DataFrame(movie_ids)
years_df = pd.DataFrame(years)
titles_df = pd.DataFrame(titles)
guides_df = pd.DataFrame(guides)
ratings_df = pd.DataFrame(ratings)

df = pd.concat([ids_df, years_df, titles_df, guides_df, ratings_df], axis=1)
df.columns = 'Movie ID', 'Year', 'Title', "Parent's Guide", 'Rating'

print(df)

    Movie ID  Year               Title Parent's Guide Rating
0  tt0106977  1993        The Fugitive          PG-13    7.8
1  tt0120616  1999           The Mummy          PG-13    7.1
2  tt0056172  1962  Lawrence of Arabia       Approved    8.3


https://www.geeksforgeeks.org/software-testing/understanding-no-such-element-exception-in-selenium/

All reviews start from 1998, independent of the movie release, and continue up to 2003. When I try to include many movies in my own list, Jupyter stops displaying output. So for now, there are 3 titles.


In [146]:
from selenium.common.exceptions import NoSuchElementException

data = []
driver.implicitly_wait(10)

for movie_id, movie_name in zip(movie_ids, titles):
    driver.get(f"https://www.imdb.com/title/{movie_id}/reviews/?sort=submission_date%2Casc&dir=asc")
    # we are already at the "reviews" page, sorted by review date in ascending order, e.g., from 1998 onward
    
    count = 1 # initialize the count for review articles

    # 1. Keep clicking "Load More" until all reviews up to 200X are loaded
    # Loop through articles until the last review after clicking "See All" multiple times, which has the year of 200X - we stop there
    while True:
        try: # Using NoSuchElementException to check if there is "No more 'Load More' button"
            see_all_button = driver.find_element(By.CSS_SELECTOR, ".ipc-see-more__button")
            previous_count = len(driver.find_elements(By.XPATH, "//article")) #
            driver.execute_script("arguments[0].click();", see_all_button) # .click() doesn't work, so we .execute_script()

            # Wait until new reviews actually appear in the DOM
            while len(driver.find_elements(By.XPATH, "//article")) == previous_count: #
                time.sleep(1) # wait for new reviews to load before clicking again

            # Check the last loaded review's year
            all_reviews = driver.find_elements(By.XPATH, "//article") 
            last_date = all_reviews[-1].find_element(By.XPATH, ".//div[2]/ul/li[2]").text # form: Sep 5, 1998
            last_year = int(last_date.split()[-1].strip()) # check the year

            if last_year > 2002:
                print(f"Loaded up to {last_year} for {movie_name}, stopping clicks")
                break

        except NoSuchElementException:
            print(f"No more 'Load More' button for {movie_name}")
            break

    # 2. Scrape all loaded reviews up to 200X
    while True:
        try:
            review = driver.find_element(By.XPATH, f"//article[{count}]")
            try:
                review_date = review.find_element(By.XPATH, ".//div[2]/ul/li[2]").text # find the review date first
                year = int(review_date.split()[-1].strip()) # check the year, should be 

                if year > 2002:
                    print(f"Reached {year} for {movie_name}, moving to next movie")
                    break

                review_text = review.find_element(By.CSS_SELECTOR, ".ipc-overflowText--children").text
                data.append({
                    'Movie': movie_name,
                    'Date': review_date,
                    'Review': review_text
                })
            except NoSuchElementException:
                print(f"Skipping review {count} — missing date or text") # It is most likely SPOILER reviews, doesn't display any text
            count += 1 # adding increment!

        except NoSuchElementException:
            print(f"No more reviews for {movie_name}")
            break

print(data)

Loaded up to 2004 for The Fugitive, stopping clicks
Skipping review 51 — missing date or text
Skipping review 74 — missing date or text
Skipping review 75 — missing date or text
Reached 2003 for The Fugitive, moving to next movie
Loaded up to 2003 for The Mummy, stopping clicks
Skipping review 273 — missing date or text
Skipping review 413 — missing date or text
Skipping review 588 — missing date or text
Skipping review 612 — missing date or text
Skipping review 660 — missing date or text
Skipping review 683 — missing date or text
Skipping review 717 — missing date or text
Skipping review 728 — missing date or text
Reached 2003 for The Mummy, moving to next movie
Loaded up to 2003 for Lawrence of Arabia, stopping clicks
Skipping review 49 — missing date or text
Skipping review 64 — missing date or text
Skipping review 122 — missing date or text
Skipping review 143 — missing date or text
Skipping review 145 — missing date or text
Reached 2003 for Lawrence of Arabia, moving to next movie

In [148]:
data_df = pd.DataFrame(data)
print(data_df)

                  Movie          Date  \
0          The Fugitive   Sep 5, 1998   
1          The Fugitive  Sep 30, 1998   
2          The Fugitive  Nov 11, 1998   
3          The Fugitive  Nov 18, 1998   
4          The Fugitive  Nov 26, 1998   
..                  ...           ...   
949  Lawrence of Arabia   Nov 5, 2002   
950  Lawrence of Arabia  Nov 11, 2002   
951  Lawrence of Arabia  Dec 10, 2002   
952  Lawrence of Arabia  Dec 30, 2002   
953  Lawrence of Arabia  Dec 31, 2002   

                                                Review  
0    This 1993 recreation of the famous 60's tv sho...  
1    When I was growing up, a movie was something b...  
2    Harrison Ford and Tommy Lee Jones race through...  
3    A perfect movie. The Fugitive renewed my wanin...  
4    Hands down the greatest movie I have ever seen...  
..                                                 ...  
949  From Sir David lean director of the oscar winn...  
950  I rate this movie 6 out of 10.\n\nThis is hard

In [ ]:
# How many reviews each movie has up to a certain date

for movie in data_df['Movie'].unique():
    review_number = len(data_df[data_df['Movie']== movie])
    print(f"{movie} has", review_number, "reviews.")


Source of Vader Sentiment Analysis: https://www.geeksforgeeks.org/python/python-sentiment-analysis-using-vader/

In [154]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

def sentiment_scores(sentence):
    sid_obj = SentimentIntensityAnalyzer()
    sentiment_dict = sid_obj.polarity_scores(sentence)
    
    print(f"Sentiment Scores: {sentiment_dict}")
    print(f"Negative Sentiment: {sentiment_dict['neg']*100}%")
    print(f"Neutral Sentiment: {sentiment_dict['neu']*100}%")
    print(f"Positive Sentiment: {sentiment_dict['pos']*100}%")
    
    if sentiment_dict['compound'] >= 0.05:
        print("Overall Sentiment: Positive")
    elif sentiment_dict['compound'] <= -0.05:
        print("Overall Sentiment: Negative")
    else:
        print("Overall Sentiment: Neutral")

    return sentiment_dict
    
# EXAMPLE
sentiment_scores('It gives a million reason why no one should go.')

Sentiment Scores: {'neg': 0.216, 'neu': 0.784, 'pos': 0.0, 'compound': -0.296}
Negative Sentiment: 21.6%
Neutral Sentiment: 78.4%
Positive Sentiment: 0.0%
Overall Sentiment: Negative


{'neg': 0.216, 'neu': 0.784, 'pos': 0.0, 'compound': -0.296}

In [155]:
# Create a list of the "compound" sentiment scores
sentiment_score = [sentiment_scores(review)['compound'] for review in data_df['Review']]
print(sentiment_score)

Sentiment Scores: {'neg': 0.089, 'neu': 0.68, 'pos': 0.231, 'compound': 0.9044}
Negative Sentiment: 8.9%
Neutral Sentiment: 68.0%
Positive Sentiment: 23.1%
Overall Sentiment: Positive
Sentiment Scores: {'neg': 0.0, 'neu': 0.767, 'pos': 0.233, 'compound': 0.9231}
Negative Sentiment: 0.0%
Neutral Sentiment: 76.7%
Positive Sentiment: 23.3%
Overall Sentiment: Positive
Sentiment Scores: {'neg': 0.102, 'neu': 0.748, 'pos': 0.15, 'compound': 0.7729}
Negative Sentiment: 10.2%
Neutral Sentiment: 74.8%
Positive Sentiment: 15.0%
Overall Sentiment: Positive
Sentiment Scores: {'neg': 0.065, 'neu': 0.717, 'pos': 0.218, 'compound': 0.9877}
Negative Sentiment: 6.5%
Neutral Sentiment: 71.7%
Positive Sentiment: 21.8%
Overall Sentiment: Positive
Sentiment Scores: {'neg': 0.024, 'neu': 0.848, 'pos': 0.128, 'compound': 0.743}
Negative Sentiment: 2.4%
Neutral Sentiment: 84.8%
Positive Sentiment: 12.8%
Overall Sentiment: Positive
Sentiment Scores: {'neg': 0.0, 'neu': 0.822, 'pos': 0.178, 'compound': 0.5707}


In [197]:
# Adding the new column of sentiment to our main dataframe
data_df['Separate_Sentiment_Score'] = sentiment_score
sentiment = [sentiment_scores(review)['compound'] for review in data_df['Review']]

# Adding the "Sentiment Overview" column to our dataframe
data_df['Sentiment'] = [
    'Positive' if score > 0.05 
    else 'Negative' if score < -0.05 
    else 'Neutral' 
    for score in data_df['Separate_Sentiment_Score']
]

Sentiment Scores: {'neg': 0.089, 'neu': 0.68, 'pos': 0.231, 'compound': 0.9044}
Negative Sentiment: 8.9%
Neutral Sentiment: 68.0%
Positive Sentiment: 23.1%
Overall Sentiment: Positive
Sentiment Scores: {'neg': 0.0, 'neu': 0.767, 'pos': 0.233, 'compound': 0.9231}
Negative Sentiment: 0.0%
Neutral Sentiment: 76.7%
Positive Sentiment: 23.3%
Overall Sentiment: Positive
Sentiment Scores: {'neg': 0.102, 'neu': 0.748, 'pos': 0.15, 'compound': 0.7729}
Negative Sentiment: 10.2%
Neutral Sentiment: 74.8%
Positive Sentiment: 15.0%
Overall Sentiment: Positive
Sentiment Scores: {'neg': 0.065, 'neu': 0.717, 'pos': 0.218, 'compound': 0.9877}
Negative Sentiment: 6.5%
Neutral Sentiment: 71.7%
Positive Sentiment: 21.8%
Overall Sentiment: Positive
Sentiment Scores: {'neg': 0.024, 'neu': 0.848, 'pos': 0.128, 'compound': 0.743}
Negative Sentiment: 2.4%
Neutral Sentiment: 84.8%
Positive Sentiment: 12.8%
Overall Sentiment: Positive
Sentiment Scores: {'neg': 0.0, 'neu': 0.822, 'pos': 0.178, 'compound': 0.5707}


Now, we calculate monthly compound sentiment

In [242]:
data_df['Date'] = pd.to_datetime(data_df['Date']) # converting to datetime format to make it possible to access the month

data_df.sort_values(by = 'Date') # sort by review date

# Monthly Sentiment
data_df['Monthly_Sentiment'] = data_df.groupby(data_df['Date'].dt.to_period('M'))['Separate_Sentiment_Score'].transform('mean')

print(data_df.head(10))

          Movie       Date                                             Review  \
0  The Fugitive 1998-09-05  This 1993 recreation of the famous 60's tv sho...   
1  The Fugitive 1998-09-30  When I was growing up, a movie was something b...   
2  The Fugitive 1998-11-11  Harrison Ford and Tommy Lee Jones race through...   
3  The Fugitive 1998-11-18  A perfect movie. The Fugitive renewed my wanin...   
4  The Fugitive 1998-11-26  Hands down the greatest movie I have ever seen...   
5  The Fugitive 1998-11-29  Very suspenseful! Harrison Ford and Tommy Lee ...   
6  The Fugitive 1998-12-10  The Fugitive is a very exciting film. I enjoye...   
7  The Fugitive 1998-12-26  What makes the 1993 version of "The Fugitive" ...   
8  The Fugitive 1999-01-05  Very seldom do television remakes do well on f...   
9  The Fugitive 1999-02-04  The Fugitive is smart, it is action packed and...   

   Sentiment_Score Sentiment  Monthly_Sentiment  Separate_Sentiment_Score  
0           0.9044  Positive    

In [164]:
# Gathering API to Create a New Variable: We get monthly box office data between 1998 and 2002. -- 4 years

box_office = BoxOffice(api_key="5d968797") 
monthly_box_office = []

for year in range(1998, 2003):
    for month in range(1, 13):
        monthly_data = box_office.get_monthly(year = year, month = month)
        monthly_box_office.append(monthly_data)
        time.sleep(1)

print(monthly_box_office)

[[{'Rank': '1', 'Release': 'Titanic', 'Gross': '$188,215,666', 'Theaters': '3,265', 'Total Gross': '$600,683,057', 'Release Date': 'Dec 19', 'Distributor': 'Paramount Pictures', 'Response': 'False', 'Error': 'Movie not found!'}, {'Rank': '2', 'Release': 'As Good as It Gets', 'Gross': '$59,523,768', 'Theaters': '1,837', 'Total Gross': '$148,478,011', 'Release Date': 'Dec 23', 'Distributor': 'Sony Pictures Releasing', 'Response': 'False', 'Error': 'Movie not found!'}, {'Rank': '3', 'Release': 'Good Will Hunting', 'Gross': '$49,409,331', 'Theaters': '2,203', 'Total Gross': '$138,433,435', 'Release Date': 'Dec 5', 'Distributor': 'Miramax', 'Response': 'False', 'Error': 'Movie not found!'}, {'Rank': '4', 'Release': 'Tomorrow Never Dies', 'Gross': '$42,773,740', 'Theaters': '2,807', 'Total Gross': '$125,304,276', 'Release Date': 'Dec 19', 'Distributor': 'Metro-Goldwyn-Mayer (MGM)', 'Response': 'False', 'Error': 'Movie not found!'}, {'Rank': '5', 'Release': 'Mousehunt', 'Gross': '$27,515,643'

In [174]:
target_movies = list(set([movie['Release'] for month in monthly_box_office for movie in month])) # set removes duplicates
print(target_movies) # Comedy movies in our box office leads

['Train of Life', "Mark Twain's America in", 'Cirque du Soleil: Journey of Man', 'Shadow Hours', 'Trois', 'Waco: The Rules of Engagement', 'Queen of the Damned', 'Carman: The Champion', 'The Object of My Affection', 'Out of Sight', 'Santa vs. the Snowman 3D', 'My Giant', 'Tumbleweeds', 'Last Night', 'Body Shots', 'The Impostors', 'The One', 'The Haunting', 'Friday After Next', "But I'm a Cheerleader", 'Finding North', 'Pumpkin', 'The Whole Nine Yards', 'Fiza', 'Shower', 'The Eyes of Tammy Faye', 'Punch-Drunk Love', 'The Lovers on the Bridge', 'The Spanish Prisoner', 'Whipped', 'The Pusher', "Monster's Ball", 'CyberWorld', 'What Planet Are You From?', 'The Visitors II: The Corridors of Time', 'Kehtaa Hai Dil Baar Baar', 'The Muse', 'Psycho Beach Party', 'The Last Days', 'All About Lily Chou-Chou', 'Ill Gotten Gains', 'Cruel Intentions', 'Fever Pitch', 'Eight Crazy Nights', 'The Parent Trap', 'Drumline', 'Hannibal', 'Tully', 'Code Unknown', 'The Boondock Saints', 'The Boxer', 'Ayn Rand: 

In [183]:
# This is so that IMDb allows us to perform stuff and doesn't block us (hinted by Jupyter helper)
import random
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--window-size=1900,1000")
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option("useAutomationExtension", False)
chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=chrome_options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

# --------------

comedy_url = "https://www.imdb.com/search/title/?title_type=feature&release_date=1998-01-01,2005-12-31&genres=comedy&moviemeter=,20000"
# we look for comedies released between 1998 and 2005 and are in the top 20,000 popular films on IMDb
driver.get(comedy_url)
time.sleep(3)

while True:
    try:
        load_more_btn = driver.find_element(By.CSS_SELECTOR, ".ipc-see-more__button") # Now trying to find the "50 more" button
        driver.execute_script("arguments[0].click();", load_more_btn) # Clicking it using JavaScript (to avoid "Intercepted" errors)
        time.sleep(3) # wait
        
    except NoSuchElementException:
        print("All Movies Are Loaded")
        break

comedies_on_page = driver.find_elements(By.XPATH, "/html/body/div[2]/main/div[2]/div[3]/section/section/div/section/section/div[2]/div/section/div[2]/div[2]/ul/li")

found = False

top_comedies = []

for item in comedies_on_page:
    card_text = item.text #
    
    for target_movie in target_movies:
        if target_movie in card_text:
            print(f"{target_movie} is a Box Office Hit.")
            top_comedies.append(target_movie)
        
            found = True
            break

Clicked '50 more' button 1 times...
Clicked '50 more' button 2 times...
Clicked '50 more' button 3 times...
Clicked '50 more' button 4 times...
Clicked '50 more' button 5 times...
Clicked '50 more' button 6 times...
Clicked '50 more' button 7 times...
Clicked '50 more' button 8 times...
Clicked '50 more' button 9 times...
Clicked '50 more' button 10 times...
Clicked '50 more' button 11 times...
Button is gone. All comedies are now loaded on the screen!
Rat is a Box Office Hit.
Rat is a Box Office Hit.
The Big Lebowski is a Box Office Hit.
Rat is a Box Office Hit.
Rat is a Box Office Hit.
Snatch is a Box Office Hit.
Rat is a Box Office Hit.
Rat is a Box Office Hit.
Wet Hot American Summer is a Box Office Hit.
Rat is a Box Office Hit.
Pi is a Box Office Hit.
Toy Story 2 is a Box Office Hit.
Shrek is a Box Office Hit.
Woo is a Box Office Hit.
Rat is a Box Office Hit.
Rat is a Box Office Hit.
O is a Box Office Hit.
Rat is a Box Office Hit.
Rat is a Box Office Hit.
Monsters, Inc. is a Box O

In [184]:
unique_comedies = set(top_comedies)
print(unique_comedies)

{'Notting Hill', 'Rushmore', '24 Hour Party People', 'The Royal Tenenbaums', 'Analyze That', 'The Ring', 'Igby Goes Down', 'The New Guy', 'Anywhere But Here', 'Dogma', 'Rat Race', "Say It Isn't So", 'Asterix and Obelix vs. Caesar', 'Topsy-Turvy', 'Out of Sight', 'Birthday Girl', 'Stuart Little 2', 'Blast from the Past', 'Rain', 'Friday After Next', 'The Wedding Singer', 'A Night at the Roxbury', "But I'm a Cheerleader", 'Detroit Rock City', 'Together', 'Forces of Nature', 'Monsters, Inc.', 'Rat', 'Men in Black II', 'Spy Kids', "A Midsummer Night's Dream", 'The Whole Nine Yards', 'Big Trouble', 'Bowfinger', 'The Mexican', 'Eight Legged Freaks', 'Snow Dogs', 'Max', 'Brother', 'Punch-Drunk Love', 'Small Time Crooks', 'Confessions of a Dangerous Mind', 'Shakespeare in Love', 'Wet Hot American Summer', 'Van Wilder', 'Pokémon the Movie 2000', 'Mean Machine', 'Miss Congeniality', 'Keeping the Faith', '40 Days and 40 Nights', 'S1m0ne', 'Wasabi', 'Almost Famous', 'Idle Hands', "Charlie's Angels

We're going to count how many comedy movies ranked each month

And divide by the total number of ranks to reflect the proportion of happy consumer sentiment.

Steps

1. Have all the months from 1998 to 2000.
2. Number the months in the list.
3. Set the consecutive numbers according to months from 1998 to 2002.
4. Find which number/month the target comedy corresponds to.
5. Count target comedies for each month.

In [273]:
# API data is the List of Lists
# monthly_box_office = [[{'Release': 'Titanic'...}], [{'Release': 'The Wedding Singer'...}], ...] -- goes like this

for index, month_data in enumerate(monthly_box_office): # 'enumerate' gives us a 'number' (index) starting at 0, and the 'month_data' (the inner list)
    current_month_label = timeline_labels[index]
    comedy_count = 0
    
    for movie in month_data:
        if movie['Release'] in unique_comedies:
            comedy_count += 1

comedy_df = pd.DataFrame(data_for_df, columns=['Month Year', 'Number of Comedies'])

print(comedy_df) # first 10 lines

   Month Year  Number of Comedies
0    Jan 1998                   0
1    Feb 1998                   2
2    Mar 1998                   3
3    Apr 1998                   1
4    May 1998                   2
5    Jun 1998                   3
6    Jul 1998                   5
7    Aug 1998                   4
8    Sep 1998                   4
9    Oct 1998                   3
10   Nov 1998                   8
11   Dec 1998                   9
12   Jan 1999                  10
13   Feb 1999                  13
14   Mar 1999                  16
15   Apr 1999                  20
16   May 1999                  17
17   Jun 1999                  15
18   Jul 1999                  12
19   Aug 1999                  10
20   Sep 1999                   8
21   Oct 1999                   9
22   Nov 1999                   6
23   Dec 1999                   8
24   Jan 2000                   8
25   Feb 2000                   8
26   Mar 2000                   8
27   Apr 2000                   9
28   May 2000 

We join two dataframes: Monthly Sentiment from the data dataframe and the Comedy Count dataframe.

But this is tricky. I don't know how to have only one value of monthly sentiment for month.

In [268]:
monthly_sentiment_df = pd.concat([comedy_df, data_df['Monthly_Sentiment']], axis = 1)

print(monthly_sentiment_df)

        Month  Comedy_Count  Monthly_Sentiment
0    Jan 1998           0.0           0.918300
1    Feb 1998           2.0           0.918300
2    Mar 1998           3.0           0.768575
3    Apr 1998           1.0           0.768575
4    May 1998           2.0           0.768575
..        ...           ...                ...
949       NaN           NaN           0.981167
950       NaN           NaN           0.981167
951       NaN           NaN           0.732418
952       NaN           NaN           0.732418
953       NaN           NaN           0.732418

[954 rows x 3 columns]


In [ ]:
driver.quit()

**I want to do a time-series regression on the data and analyze whether there was a change in the monthly sentiment of reviews of certain films after September 2001.**
